# Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')
%cd /content/drive/MyDrive/spinal-bone-feature-detection
!ls

In [ ]:
!unzip -q ./datasets/ultrasound.zip -d /tmp/ultrasound
!pip install -q torchmetrics[detection]
# !pip install -q torch_geometric

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
from torchvision.ops import box_iou
from torchvision.models import resnet18
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from sklearn.metrics import classification_report, precision_recall_curve, confusion_matrix, roc_curve, auc
from scipy.optimize import linear_sum_assignment
from scipy.interpolate import CubicSpline

import gc
import yaml
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image
from trainers import FasterRCNNTrainer
from loader import ScoliosisDataset, get_loader, concat_batch

In [ ]:
%load_ext tensorboard
!rm -rf results
plt.style.use('seaborn-v0_8')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
splits_path = 'kfolds/splits1.yaml'

# Model Experiment

In [ ]:
train_loader = get_loader('train', splits_path=splits_path, batch_size=4)
val_loader = get_loader('val', splits_path=splits_path, batch_size=1)
for names, images, targets in train_loader:
    print(f'Batch images shape: {images.shape}')
    for names, target in zip(names, targets):
        print(f"{names}: boxes={target['boxes'].shape}, labels={target['labels'].shape}")
    break

In [ ]:
def get_resnet18_backbone():
    backbone = resnet18()
    original_conv1 = backbone.conv1
    backbone = torch.nn.Sequential(*list(backbone.children())[:-2]) # Remove the classification head
    backbone[0] = nn.Conv2d( # Modify the first conv layer to accept 2 channels
        in_channels=2,
        out_channels=original_conv1.out_channels,
        kernel_size=original_conv1.kernel_size,
        stride=original_conv1.stride,
        padding=original_conv1.padding,
        bias=False
    )
    backbone.out_channels = 512 # Define number of output channels (ResNet-18's final layer before avgpool has 512)
    return backbone

In [ ]:
model = FasterRCNN(
    get_resnet18_backbone(), num_classes=3, # 2 classes (Thoracic, Lumbar) + background
    rpn_anchor_generator=AnchorGenerator(
        sizes=((32, 64, 128, 256, 512),),
        aspect_ratios=((0.5, 1.0, 2.0),)
    )
).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

In [ ]:
temp_dataset = ScoliosisDataset(split='train', config_path='config.yaml', splits_path=splits_path)
dataset_mean = temp_dataset.means
dataset_std = temp_dataset.stds

# Update the model's internal transform with the correct mean and std
# Access the transform through model.transform
model.transform.image_mean = dataset_mean.tolist()
model.transform.image_std = dataset_std.tolist()

In [ ]:
%%time
gc.collect()
torch.cuda.empty_cache()
trainer = FasterRCNNTrainer(model, train_loader, val_loader, num_epochs=200, optimizer=optimizer)
trainer.train()

In [ ]:
%tensorboard --logdir results

# Evaluation

## mAP, Cls Report, PR Curve

In [ ]:
checkpoint = torch.load('best_model.pth')
trainer.model.load_state_dict(checkpoint)
mAP_preds, mAP_targets, results = trainer.evaluate()
results

In [ ]:
def compute_pr_curve(preds, targets, iou_threshold=0.5): # Evaluate predictions and compute PR curve
    all_scores, all_labels = [], [] # 1 for TP, 0 for FP
    all_matched_pred_labels, all_matched_gt_labels = [], []

    for pred, target in zip(preds, targets):
        pred_boxes = pred['boxes']   # [N, 4]
        scores = pred['scores']      # [N]
        pred_labels = pred['labels'] # [N]
        gt_boxes = target['boxes']   # [M, 4]
        gt_labels = target['labels'] # [M]

        matched_pred_labels, matched_gt_labels = [], []
        if len(pred_boxes) == 0 or len(gt_boxes) == 0: continue
        used_gt = set()

        # Compute IoU between all pred and gt boxes
        # iou = box_iou(pred_boxes, gt_boxes).cpu().numpy()  # [N, M]
        # max_iou = np.max(iou, axis=1)  # Best IoU for each prediction
        # matched_idx = np.argmax(iou, axis=1)
        ious = box_iou(pred_boxes, gt_boxes)  # shape: [num_preds, num_gts]

        # For each prediction, determine if it's a TP or FP
        for i, (score, pred_label) in enumerate(zip(scores, pred_labels)):
            iou_vals = ious[i]
            max_iou, gt_idx = torch.max(iou_vals, dim=0)

            if max_iou >= iou_threshold and gt_idx.item() not in used_gt:
                if pred_label == gt_labels[gt_idx]:
                    matched_pred_labels.append(pred_labels[i].item())
                    matched_gt_labels.append(gt_labels[gt_idx].item())
                    used_gt.add(gt_idx)
                    all_labels.append(1)  # True Positive
                else:
                    matched_pred_labels.append(pred_labels[i].item())
                    matched_gt_labels.append(0)  # or a dummy class for "no object"
                    all_labels.append(0)  # False Positive (wrong label)
            else:
                matched_pred_labels.append(pred_labels[i].item())
                matched_gt_labels.append(0)  # or a dummy class for "no object"
                all_labels.append(0)  # False Positive (no match)
            all_scores.append(score.cpu().numpy())

        for i in range(len(gt_boxes)):
            if i not in used_gt:
                matched_pred_labels.append(0)  # or a dummy class for "no object"
                matched_gt_labels.append(gt_labels[i].item())
        all_matched_pred_labels.extend(matched_pred_labels)
        all_matched_gt_labels.extend(matched_gt_labels)


    # Sort by scores in descending order
    all_scores = np.array(all_scores)
    all_labels = np.array(all_labels)
    order = np.argsort(all_scores)[::-1]
    all_scores = all_scores[order]
    all_labels = all_labels[order]

    # Compute precision and recall
    precision, recall, _ = precision_recall_curve(all_labels, all_scores)
    return precision, recall, all_matched_gt_labels, all_matched_pred_labels

In [ ]:
label_names=['Thoracic', 'Lumbar']
precision, recall, all_matched_gt_labels, all_matched_pred_labels = compute_pr_curve(mAP_preds, mAP_targets)
print(classification_report(all_matched_gt_labels, all_matched_pred_labels, target_names=label_names))
plt.plot(recall, precision, marker='.')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.grid(True)
plt.show()

## Display Validation Inference

In [ ]:
checkpoint = torch.load('best_model.pth')
model.load_state_dict(checkpoint)
model.eval()

with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)
    images_dir = Path(config['data']['images_dir'])
    processed_height, processed_width = config['preprocessing']['image_size']

def get_scale(image):
    orig_width, orig_height = image.size
    scale_x = orig_width / processed_width
    scale_y = orig_height / processed_height
    return scale_x, scale_y

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=8, figsize=(15, 10))
axes = axes.flatten()  # Flatten the grid for easy indexing
for ax in axes: ax.grid(False) # Turn off gridlines
grid_idx, pred_prid_idx = 0, 0 # Counter for the grid positions

for names, images, targets in val_loader:
    images, targets, boxes_by_images, labels_by_images, all_boxes, all_labels = concat_batch(images, targets, device)
    outputs = model(images)

    # Get the number of predicted boxes for each image in the batch
    pred_lengths = [len(output['boxes']) for output in outputs]
    pred_boxes = torch.cat([output['boxes'] for output in outputs])
    pred_labels = torch.cat([output['labels'] for output in outputs])
    scores = torch.cat([output['scores'] for output in outputs])

    # Reshape to match the number of predicted boxes per image
    pred_boxes_by_images = torch.split(pred_boxes, pred_lengths)
    pred_labels_by_images = torch.split(pred_labels, pred_lengths)
    scores_by_images = torch.split(scores, pred_lengths)

    # Plot each ground truth box
    for name, boxes, labels in zip(names, boxes_by_images, labels_by_images):
        if grid_idx >= len(axes): break # Stop if all subplots are filled
        image = Image.open(images_dir / f'{name}.jpg')
        scale_x, scale_y = get_scale(image) # Scale the boxes back to the original image dimensions
        orig_width, orig_height = image.size

        # Display image
        axes[grid_idx].imshow(image, cmap='gray')
        axes[grid_idx].axis('off') # Turn off axis labels and ticks
        axes[grid_idx].set_title(f'{orig_height} x {orig_width}\n({len(boxes)} boxes detected)')
        grid_idx += 1  # Move to the next subplot position

        for box, label in zip(boxes, labels):
            if label >= 2: continue
            x1, y1, x2, y2 = box.cpu().numpy()
            x1, y1, x2, y2 = x1 * scale_x, y1 * scale_y, x2 * scale_x, y2 * scale_y
            color = 'red' if label == 0 else 'green'
            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=1, edgecolor=color, facecolor='none')
            axes[grid_idx - 1].add_patch(rect) # Add the rectangle to the plot

    # Plot each predicted box (Similar processing as for ground truth boxes)
    for name, boxes, labels, scores in zip(names, pred_boxes_by_images, pred_labels_by_images, scores_by_images):
        if pred_prid_idx >= len(axes): break # Stop if all subplots are filled
        image = Image.open(images_dir / f'{name}.jpg')
        scale_x, scale_y = get_scale(image) # Scale the boxes back to the original image dimensions
        pred_prid_idx += 1  # Move to the next subplot position

        for box, label, score in zip(boxes, labels, scores):
            if label >= 2: continue
            x1, y1, x2, y2 = box.cpu().detach().numpy() # Detach the tensor before converting to NumPy
            x1, y1, x2, y2 = x1 * scale_x, y1 * scale_y, x2 * scale_x, y2 * scale_y
            color = 'cyan' if label == 0 else 'gold'
            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=1, edgecolor=color, facecolor='none')
            axes[pred_prid_idx - 1].add_patch(rect) # Add the rectangle to the plot
            axes[pred_prid_idx - 1].text(
                x1, y1, f'{score.cpu().detach().numpy():.4f}',
                fontsize=9, fontweight='bold',
                color=color, # bbox=dict(facecolor='white', alpha=0.2)
            )
    if grid_idx >= len(axes): break # Stop if all subplots are filled

for ax in axes[grid_idx:]: ax.axis('off') # Hide any unused subplots
plt.tight_layout()
plt.show()